# Financial linking and abnormal return extraction

This notebook takes the completed textual-analysis output and the daily stock return data, then:

1. loads and cleans the three input files
2. links each earnings call to a `PERMNO` using `ticker`
3. computes daily abnormal returns as `DlyRet - vwretd`
4. extracts short-run `CAR[-1,+3]` around each earnings call
5. extracts a longer-run abnormal return window up to the next earnings call
6. merges the financial variables back into the textual-analysis table
7. exports a regression-ready dataset

**Input files expected in the same folder as this notebook**
- `textual_analysis_results.csv`
- `daily_ret.csv`
- `linking_table.csv`

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. File paths

In [2]:
from pathlib import Path

BASE = Path("../../data")

TA_RES_PATH = BASE / "processed/textual_analysis_results.csv"
RET_PATH = BASE / "raw/daily_ret.csv"
LINK_PATH = BASE / "raw/linking_table.csv"

print(TA_RES_PATH.resolve())
print(RET_PATH.resolve())
print(LINK_PATH.resolve())
print("TA exists:", TA_RES_PATH.exists())
print("RET exists:", RET_PATH.exists())
print("LINK exists:", LINK_PATH.exists())

/Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/processed/textual_analysis_results.csv
/Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/raw/daily_ret.csv
/Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/raw/linking_table.csv
TA exists: True
RET exists: True
LINK exists: True


## 2. Load datasets

Note: `ai_counts.csv` is semicolon-separated, and several sentiment/intensity variables use commas as decimal separators.

In [3]:
ta = pd.read_csv(TA_RES_PATH, sep=";")
ret = pd.read_csv(RET_PATH)
link = pd.read_csv(LINK_PATH)

print("ta shape  :", ta.shape)
print("ret shape :", ret.shape)
print("link shape:", link.shape)

display(ta.head())
display(ret.head())
display(link.head())

ta shape  : (96, 37)
ret shape : (102309, 11)
link shape: (102, 7)


,company,ticker,date,file,total_words,pres_words,qa_words,core_hits_total,core_hits_pres,core_hits_qa,adj_hits_total,adj_hits_pres,adj_hits_qa,lm_positive_total,lm_negative_total,lm_uncertainty_count_total,lm_tone_total,lm_uncertainty_total,lm_positive_pres,lm_negative_pres,lm_uncertainty_count_pres,lm_tone_pres,lm_uncertainty_pres,lm_positive_qa,lm_negative_qa,lm_uncertainty_count_qa,lm_tone_qa,lm_uncertainty_qa,fiscal_quarter,fiscal_year,fiscal_period,core_per_1000_total,core_per_1000_pres,core_per_1000_qa,adj_per_1000_total,adj_per_1000_pres,adj_per_1000_qa
0,Alphabet Inc,GOOGL,2022-04-26,2022-Apr-26-GOOGL.OQ-138254363287-Transcript.txt,8430,4250,4020,9,9,0,12,8,4,135,70,55,"0,3170731707317073","0,006871564217891054",83,20,17,"0,6116504854368932","0,004216269841269841",52,49,38,"0,0297029702970297","0,009903570497784727",1,2022,Q1 2022,"1,0676156583629894","2,1176470588235294","0,0","1,4234875444839858","1,8823529411764706","0,9950248756218905"
1,Alphabet Inc,GOOGL,2022-02-01,2022-Feb-01-GOOGL.OQ-138419589371-Transcript.txt,9563,4214,5189,32,16,16,13,7,6,170,64,58,"0,452991452991453","0,006384149697303247",83,15,16,"0,6938775510204082","0,004041424602172266",87,48,42,"0,28888888888888886","0,008415147265077139",4,2021,Q4 2021,"3,3462302624699363","3,7968675842429995","3,083445750626325","1,3594060441284115","1,6611295681063123","1,1562921564848718"
2,Alphabet Inc,GOOGL,2022-07-26,2022-Jul-26-GOOGL.OQ-140228385298-Transcript.txt,8594,3955,4468,22,14,8,7,5,2,143,63,75,"0,3883495145631068","0,009119649805447471",66,20,18,"0,5348837209302325","0,004805125467164976",77,42,57,"0,29411764705882354","0,013163972286374134",2,2022,Q2 2022,"2,559925529439144","3,5398230088495577","1,7905102954341987","0,8145217593670003","1,2642225031605563","0,4476275738585497"
3,Alphabet Inc,GOOGL,2022-10-25,2022-Oct-25-GOOGL.OQ-139231849236-Transcript.txt,8954,4277,4511,20,11,9,10,6,4,134,72,62,"0,30097087378640774","0,007222739981360671",68,29,15,"0,4020618556701031","0,0036836935166994107",66,42,47,"0,2222222222222222","0,01076007326007326",3,2022,Q3 2022,"2,233638597274961","2,5718961889174654","1,9951230325870095","1,1168192986374805","1,4028524666822537","0,8867213478164486"
4,Alphabet Inc,GOOGL,2023-04-25,2023-Apr-25-GOOGL.OQ-140631444702-Transcript.txt,9229,4418,4649,93,64,29,11,7,4,196,51,64,"0,5870445344129555","0,007221846084405326",110,16,15,"0,746031746031746","0,0035486160397444995",86,34,49,"0,43333333333333335","0,010898576512455516",1,2023,Q1 2023,"10,07693141185394","14,48619284744228","6,237900623790062","1,1918951132300357","1,5844273426889997","0,8604000860400086"


,PERMNO,HdrCUSIP,Ticker,PERMCO,IssuerNm,DlyCalDt,DlyPrc,DlyRet,DlyVol,vwretd,sprtrn
0,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-03,87.90,0.007912,10653484,0.006155,0.006374
1,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-04,88.84,0.010694,11958975,-0.002351,-0.000630
2,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-05,86.46,-0.026790,11236680,-0.021879,-0.019393
3,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-06,86.34,0.002313,7918392,0.000161,-0.000964
4,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-07,87.51,0.013551,9836795,-0.004164,-0.004050


,PERMNO,HdrCUSIP,CUSIP,Ticker,PERMCO,IssuerNm,DlyCalDt
0,10104,68389X10,68389X10,ORCL,8045,ORACLE CORP,2025-12-31
1,10107,59491810,59491810,MSFT,8048,MICROSOFT CORP,2025-12-31
2,10145,43851610,43851610,HON,22168,HONEYWELL INTERNATIONAL INC,2025-12-31
3,11308,19121610,19121610,KO,20468,COCA COLA CO,2025-12-31
4,11850,30231G10,30231G10,XOM,20678,EXXON MOBIL CORP,2025-12-31


## 3. Basic cleaning

In [4]:
# Dates
ta["date"] = pd.to_datetime(ta["date"], errors="coerce")
ret["DlyCalDt"] = pd.to_datetime(ret["DlyCalDt"], errors="coerce")

# Convert comma-decimal columns in textual_analysis_results.csv to numeric
comma_decimal_cols = [
    "lm_tone_total", "lm_uncertainty_total",
    "lm_tone_pres", "lm_uncertainty_pres",
    "lm_tone_qa", "lm_uncertainty_qa",
    "core_per_1000_total", "core_per_1000_pres", "core_per_1000_qa",
    "adj_per_1000_total", "adj_per_1000_pres", "adj_per_1000_qa",
]

for col in comma_decimal_cols:
    if col in ta.columns:
        ta[col] = (
            ta[col]
            .astype(str)
            .str.replace(",", ".", regex=False)
            .replace("nan", np.nan)
            .astype(float)
        )

# Ticker cleanup
ta["ticker"] = ta["ticker"].astype(str).str.upper().str.strip()
link["Ticker"] = link["Ticker"].astype(str).str.upper().str.strip()

# PERMNO / returns cleanup
ret["PERMNO"] = pd.to_numeric(ret["PERMNO"], errors="coerce").astype("Int64")
link["PERMNO"] = pd.to_numeric(link["PERMNO"], errors="coerce").astype("Int64")
ret["DlyRet"] = pd.to_numeric(ret["DlyRet"], errors="coerce")
ret["vwretd"] = pd.to_numeric(ret["vwretd"], errors="coerce")

print(ta.dtypes.head(20))

company                               object
ticker                                object
date                          datetime64[ns]
file                                  object
total_words                            int64
pres_words                             int64
qa_words                               int64
core_hits_total                        int64
core_hits_pres                         int64
core_hits_qa                           int64
adj_hits_total                         int64
adj_hits_pres                          int64
adj_hits_qa                            int64
lm_positive_total                      int64
lm_negative_total                      int64
lm_uncertainty_count_total             int64
lm_tone_total                        float64
lm_uncertainty_total                 float64
lm_positive_pres                       int64
lm_negative_pres                       int64
dtype: object


## 4. XXX

In [5]:
# Keep one row per ticker/PERMNO pair in link table
link_unique = (
    link[["PERMNO", "Ticker", "IssuerNm"]]
    .drop_duplicates()
    .copy()
)

display(link_unique.head())

,PERMNO,Ticker,IssuerNm
0,10104,ORCL,ORACLE CORP
1,10107,MSFT,MICROSOFT CORP
2,10145,HON,HONEYWELL INTERNATIONAL INC
3,11308,KO,COCA COLA CO
4,11850,XOM,EXXON MOBIL CORP


## 5. Link earnings calls to PERMNO

In [6]:
ta_linked = ta.merge(
    link_unique,
    left_on="ticker",
    right_on="Ticker",
    how="left"
)

print("Rows without PERMNO after ticker merge:", ta_linked["PERMNO"].isna().sum())
display(
    ta_linked[ta_linked["PERMNO"].isna()][["company", "ticker", "file"]].drop_duplicates()
)
display(ta_linked.head())

Rows without PERMNO after ticker merge: 0


,company,ticker,file


,company,ticker,date,file,total_words,pres_words,qa_words,core_hits_total,core_hits_pres,core_hits_qa,adj_hits_total,adj_hits_pres,adj_hits_qa,lm_positive_total,lm_negative_total,lm_uncertainty_count_total,lm_tone_total,lm_uncertainty_total,lm_positive_pres,lm_negative_pres,lm_uncertainty_count_pres,lm_tone_pres,lm_uncertainty_pres,lm_positive_qa,lm_negative_qa,lm_uncertainty_count_qa,lm_tone_qa,lm_uncertainty_qa,fiscal_quarter,fiscal_year,fiscal_period,core_per_1000_total,core_per_1000_pres,core_per_1000_qa,adj_per_1000_total,adj_per_1000_pres,adj_per_1000_qa,PERMNO,Ticker,IssuerNm
0,Alphabet Inc,GOOGL,2022-04-26,2022-Apr-26-GOOGL.OQ-138254363287-Transcript.txt,8430,4250,4020,9,9,0,12,8,4,135,70,55,0.317073,0.006872,83,20,17,0.611650,0.004216,52,49,38,0.029703,0.009904,1,2022,Q1 2022,1.067616,2.117647,0.000000,1.423488,1.882353,0.995025,90319,GOOGL,ALPHABET INC
1,Alphabet Inc,GOOGL,2022-02-01,2022-Feb-01-GOOGL.OQ-138419589371-Transcript.txt,9563,4214,5189,32,16,16,13,7,6,170,64,58,0.452991,0.006384,83,15,16,0.693878,0.004041,87,48,42,0.288889,0.008415,4,2021,Q4 2021,3.346230,3.796868,3.083446,1.359406,1.661130,1.156292,90319,GOOGL,ALPHABET INC
2,Alphabet Inc,GOOGL,2022-07-26,2022-Jul-26-GOOGL.OQ-140228385298-Transcript.txt,8594,3955,4468,22,14,8,7,5,2,143,63,75,0.388350,0.009120,66,20,18,0.534884,0.004805,77,42,57,0.294118,0.013164,2,2022,Q2 2022,2.559926,3.539823,1.790510,0.814522,1.264223,0.447628,90319,GOOGL,ALPHABET INC
3,Alphabet Inc,GOOGL,2022-10-25,2022-Oct-25-GOOGL.OQ-139231849236-Transcript.txt,8954,4277,4511,20,11,9,10,6,4,134,72,62,0.300971,0.007223,68,29,15,0.402062,0.003684,66,42,47,0.222222,0.010760,3,2022,Q3 2022,2.233639,2.571896,1.995123,1.116819,1.402852,0.886721,90319,GOOGL,ALPHABET INC
4,Alphabet Inc,GOOGL,2023-04-25,2023-Apr-25-GOOGL.OQ-140631444702-Transcript.txt,9229,4418,4649,93,64,29,11,7,4,196,51,64,0.587045,0.007222,110,16,15,0.746032,0.003549,86,34,49,0.433333,0.010899,1,2023,Q1 2023,10.076931,14.486193,6.237901,1.191895,1.584427,0.860400,90319,GOOGL,ALPHABET INC


Sanity check to see if each ticker really only equals 1 PERMNO

In [16]:
print("Unique tickers in textual results:", ta["ticker"].nunique())
print("Unique tickers matched to PERMNO:", ta_linked.loc[ta_linked["PERMNO"].notna(), "ticker"].nunique())

display(
    ta_linked.groupby("ticker", dropna=False)["PERMNO"]
    .nunique()
    .sort_values(ascending=False)
    .head(20)
)

Unique tickers in textual results: 6
Unique tickers matched to PERMNO: 6


ticker
AAPL     1
AMZN     1
GOOGL    1
META     1
MSFT     1
TSLA     1
Name: PERMNO, dtype: int64

## 6. Compute daily abnormal return

Using your chosen market-adjusted specification:

\[
AR_{i,t} = DlyRet_{i,t} - vwretd_t
\]

In [7]:
ret = ret.copy()
ret["abret"] = ret["DlyRet"] - ret["vwretd"]

# Keep only what is needed
ret_small = (
    ret[["PERMNO", "Ticker", "IssuerNm", "DlyCalDt", "DlyRet", "vwretd", "abret"]]
    .sort_values(["PERMNO", "DlyCalDt"])
    .reset_index(drop=True)
)

display(ret_small.head())

,PERMNO,Ticker,IssuerNm,DlyCalDt,DlyRet,vwretd,abret
0,10104,ORCL,ORACLE CORP,2022-01-03,0.007912,0.006155,0.001757
1,10104,ORCL,ORACLE CORP,2022-01-04,0.010694,-0.002351,0.013045
2,10104,ORCL,ORACLE CORP,2022-01-05,-0.026790,-0.021879,-0.004911
3,10104,ORCL,ORACLE CORP,2022-01-06,0.002313,0.000161,0.002152
4,10104,ORCL,ORACLE CORP,2022-01-07,0.013551,-0.004164,0.017715


## 7. Prepare event table

Sort each firm's earnings calls so we can identify the next call date for the long-run window.

In [8]:
events = ta_linked.copy()
events = events.sort_values(["PERMNO", "date"]).reset_index(drop=True)
events["next_call_date"] = events.groupby("PERMNO")["date"].shift(-1)

display(events[["company", "ticker", "date", "PERMNO", "next_call_date"]].head(10))

,company,ticker,date,PERMNO,next_call_date
0,Microsoft Corp,MSFT,2022-01-25,10107,2022-04-26
1,Microsoft Corp,MSFT,2022-04-26,10107,2022-07-26
2,Microsoft Corp,MSFT,2022-07-26,10107,2022-10-25
3,Microsoft Corp,MSFT,2022-10-25,10107,2023-01-24
4,Microsoft Corp,MSFT,2023-01-24,10107,2023-04-25
5,Microsoft Corp,MSFT,2023-04-25,10107,2023-07-25
6,Microsoft Corp,MSFT,2023-07-25,10107,2023-10-24
7,Microsoft Corp,MSFT,2023-10-24,10107,2024-01-30
8,Microsoft Corp,MSFT,2024-01-30,10107,2024-04-25
9,Microsoft Corp,MSFT,2024-04-25,10107,2024-07-30


## 8. Helper functions for event-window extraction

Two important choices here:

- `CAR[-1,+3]` uses **trading days**, not calendar days.
- If the earnings call date itself is not a trading day, the notebook shifts to the **next available trading day** for that firm.

In [9]:
def get_trading_dates_for_firm(df_firm):
    return df_firm["DlyCalDt"].dropna().drop_duplicates().sort_values().tolist()

def get_event_trading_day(df_firm, event_date):
    # first trading day on or after the event date
    dates = df_firm.loc[df_firm["DlyCalDt"] >= event_date, "DlyCalDt"].drop_duplicates().sort_values()
    if len(dates) == 0:
        return pd.NaT
    return dates.iloc[0]

def compute_car_minus1_plus3(df_firm, event_date):
    event_trading_day = get_event_trading_day(df_firm, event_date)
    if pd.isna(event_trading_day):
        return pd.Series({
            "event_trading_day": pd.NaT,
            "car_m1_p3": np.nan,
            "n_days_car": 0
        })

    firm_dates = df_firm["DlyCalDt"].drop_duplicates().sort_values().reset_index(drop=True)
    try:
        idx = firm_dates[firm_dates == event_trading_day].index[0]
    except IndexError:
        return pd.Series({
            "event_trading_day": pd.NaT,
            "car_m1_p3": np.nan,
            "n_days_car": 0
        })

    start_idx = max(idx - 1, 0)
    end_idx = min(idx + 3, len(firm_dates) - 1)
    window_dates = firm_dates.iloc[start_idx:end_idx + 1]

    temp = df_firm[df_firm["DlyCalDt"].isin(window_dates)].copy()

    return pd.Series({
        "event_trading_day": event_trading_day,
        "car_m1_p3": temp["abret"].sum(skipna=True),
        "n_days_car": temp["abret"].notna().sum()
    })

def compute_long_run_abret(df_firm, event_date, next_call_date):
    event_trading_day = get_event_trading_day(df_firm, event_date)
    if pd.isna(event_trading_day):
        return pd.Series({
            "long_run_start": pd.NaT,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })

    # Start the day after the event trading day
    firm_dates = df_firm["DlyCalDt"].drop_duplicates().sort_values().reset_index(drop=True)
    try:
        event_idx = firm_dates[firm_dates == event_trading_day].index[0]
    except IndexError:
        return pd.Series({
            "long_run_start": pd.NaT,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })

    start_idx = event_idx + 1
    if start_idx >= len(firm_dates):
        return pd.Series({
            "long_run_start": pd.NaT,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })

    long_run_start = firm_dates.iloc[start_idx]

    # If there is a next call, end at the trading day before that next call's event trading day
    if pd.notna(next_call_date):
        next_event_trading_day = get_event_trading_day(df_firm, next_call_date)
        if pd.notna(next_event_trading_day):
            next_idx = firm_dates[firm_dates == next_event_trading_day].index[0]
            end_idx = next_idx - 1
        else:
            end_idx = len(firm_dates) - 1
    else:
        end_idx = len(firm_dates) - 1

    if end_idx < start_idx:
        return pd.Series({
            "long_run_start": long_run_start,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })

    window_dates = firm_dates.iloc[start_idx:end_idx + 1]
    long_run_end = window_dates.iloc[-1]

    temp = df_firm[df_firm["DlyCalDt"].isin(window_dates)].copy()

    return pd.Series({
        "long_run_start": long_run_start,
        "long_run_end": long_run_end,
        "long_run_abret": temp["abret"].sum(skipna=True),
        "n_days_long_run": temp["abret"].notna().sum()
    })

## 9. Run the extraction firm by firm

In [10]:
results = []

for _, row in events.iterrows():
    permno = row["PERMNO"]
    event_date = row["date"]
    next_call_date = row["next_call_date"]

    if pd.isna(permno) or pd.isna(event_date):
        out = row.to_dict()
        out.update({
            "event_trading_day": pd.NaT,
            "car_m1_p3": np.nan,
            "n_days_car": 0,
            "long_run_start": pd.NaT,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })
        results.append(out)
        continue

    df_firm = ret_small[ret_small["PERMNO"] == permno].sort_values("DlyCalDt").copy()

    car_info = compute_car_minus1_plus3(df_firm, event_date)
    long_info = compute_long_run_abret(df_firm, event_date, next_call_date)

    out = row.to_dict()
    out.update(car_info.to_dict())
    out.update(long_info.to_dict())
    results.append(out)

financial_event_df = pd.DataFrame(results)

display(
    financial_event_df[
        ["company", "date", "PERMNO", "event_trading_day", "car_m1_p3", "n_days_car", "long_run_start", "long_run_end", "long_run_abret", "n_days_long_run"]
    ].head(15)
)

,company,date,PERMNO,event_trading_day,car_m1_p3,n_days_car,long_run_start,long_run_end,long_run_abret,n_days_long_run
0,Microsoft Corp,2022-01-25,10107,2022-01-25,0.039141,5,2022-01-26,2022-04-25,-0.002991,62
1,Microsoft Corp,2022-04-26,10107,2022-04-26,0.048218,5,2022-04-27,2022-07-25,0.017476,61
2,Microsoft Corp,2022-07-26,10107,2022-07-26,0.037095,5,2022-07-27,2022-10-24,0.022081,63
3,Microsoft Corp,2022-10-25,10107,2022-10-25,-0.061268,5,2022-10-26,2023-01-23,-0.069766,60
4,Microsoft Corp,2023-01-24,10107,2023-01-24,0.008239,5,2023-01-25,2023-04-24,0.141857,62
5,Microsoft Corp,2023-04-25,10107,2023-04-25,0.070937,5,2023-04-26,2023-07-24,0.123525,61
6,Microsoft Corp,2023-07-25,10107,2023-07-25,-0.023971,5,2023-07-26,2023-10-23,0.028720,63
7,Microsoft Corp,2023-10-24,10107,2023-10-24,0.035571,5,2023-10-25,2024-01-29,0.067357,65
8,Microsoft Corp,2024-01-30,10107,2024-01-30,0.007434,5,2024-01-31,2024-04-24,-0.022996,59
9,Microsoft Corp,2024-04-25,10107,2024-04-25,-0.037207,5,2024-04-26,2024-07-29,-0.004226,64


## 10. Quick quality checks

In [11]:
print("Missing PERMNO rows:", financial_event_df["PERMNO"].isna().sum())
print("Missing CAR rows   :", financial_event_df["car_m1_p3"].isna().sum())
print("Missing long-run rows:", financial_event_df["long_run_abret"].isna().sum())

print("\nDistribution of n_days_car")
print(financial_event_df["n_days_car"].value_counts(dropna=False).sort_index())

print("\nDistribution of n_days_long_run")
print(financial_event_df["n_days_long_run"].value_counts(dropna=False).sort_index().head(20))

display(financial_event_df[financial_event_df["n_days_car"] < 4][["company", "date", "PERMNO", "event_trading_day", "n_days_car"]].head(20))

Missing PERMNO rows: 0
Missing CAR rows   : 0
Missing long-run rows: 0

Distribution of n_days_car
n_days_car
5    96
Name: count, dtype: int64

Distribution of n_days_long_run
n_days_long_run
42     2
43     3
48     1
54     1
55     1
56     2
57     8
58     1
59     6
60     4
61    18
62     9
63    20
64     6
65     8
66     3
67     1
68     2
Name: count, dtype: int64


,company,date,PERMNO,event_trading_day,n_days_car


For `CAR[-1,+3]`, you will usually expect **5 trading days** when the firm has full return coverage for the window.  
If some rows show fewer days, inspect them manually.

## 11. Create final regression-ready dataset

In [12]:
final_cols_front = [
    "company", "ticker", "date", "file", "PERMNO", "Ticker", "IssuerNm",
    "event_trading_day", "next_call_date",
    "car_m1_p3", "n_days_car",
    "long_run_start", "long_run_end", "long_run_abret", "n_days_long_run"
]

remaining_cols = [c for c in financial_event_df.columns if c not in final_cols_front]
final_df = financial_event_df[final_cols_front + remaining_cols].copy()

display(final_df.head())
print(final_df.shape)

,company,ticker,date,file,PERMNO,Ticker,IssuerNm,event_trading_day,next_call_date,car_m1_p3,n_days_car,long_run_start,long_run_end,long_run_abret,n_days_long_run,total_words,pres_words,qa_words,core_hits_total,core_hits_pres,core_hits_qa,adj_hits_total,adj_hits_pres,adj_hits_qa,lm_positive_total,lm_negative_total,lm_uncertainty_count_total,lm_tone_total,lm_uncertainty_total,lm_positive_pres,lm_negative_pres,lm_uncertainty_count_pres,lm_tone_pres,lm_uncertainty_pres,lm_positive_qa,lm_negative_qa,lm_uncertainty_count_qa,lm_tone_qa,lm_uncertainty_qa,fiscal_quarter,fiscal_year,fiscal_period,core_per_1000_total,core_per_1000_pres,core_per_1000_qa,adj_per_1000_total,adj_per_1000_pres,adj_per_1000_qa
0,Microsoft Corp,MSFT,2022-01-25,2022-Jan-25-MSFT.OQ-140806504057-Transcript.txt,10107,MSFT,MICROSOFT CORP,2022-01-25,2022-04-26,0.039141,5,2022-01-26,2022-04-25,-0.002991,62,9526,5452,3936,5,4,1,14,7,7,153,48,79,0.522388,0.008754,99,18,40,0.692308,0.007773,54,29,39,0.301205,0.010361,2,2022,Q2 2022,0.524879,0.733676,0.254065,1.469662,1.283933,1.778455
1,Microsoft Corp,MSFT,2022-04-26,2022-Apr-26-MSFT.OQ-138905362958-Transcript.txt,10107,MSFT,MICROSOFT CORP,2022-04-26,2022-07-26,0.048218,5,2022-04-27,2022-07-25,0.017476,61,9923,5382,4399,13,5,8,12,5,7,130,76,103,0.262136,0.011001,79,40,49,0.327731,0.009736,51,35,54,0.186047,0.012827,3,2022,Q3 2022,1.310088,0.929023,1.818595,1.209312,0.929023,1.591271
2,Microsoft Corp,MSFT,2022-07-26,2022-Jul-26-MSFT.OQ-139232561311-Transcript.txt,10107,MSFT,MICROSOFT CORP,2022-07-26,2022-10-25,0.037095,5,2022-07-27,2022-10-24,0.022081,63,9703,6045,3512,9,9,0,3,3,0,112,90,81,0.108911,0.008869,83,54,48,0.211679,0.008478,29,35,33,-0.093750,0.009851,4,2022,Q4 2022,0.927548,1.488834,0.000000,0.309183,0.496278,0.000000
3,Microsoft Corp,MSFT,2022-10-25,2022-Oct-25-MSFT.OQ-137469915199-Transcript.txt,10107,MSFT,MICROSOFT CORP,2022-10-25,2023-01-24,-0.061268,5,2022-10-26,2023-01-23,-0.069766,60,9893,5655,4096,20,6,14,6,4,2,129,85,99,0.205607,0.010612,70,40,53,0.272727,0.010023,59,44,46,0.145631,0.011732,1,2023,Q1 2023,2.021631,1.061008,3.417969,0.606489,0.707339,0.488281
4,Microsoft Corp,MSFT,2023-01-24,2023-Jan-24-MSFT.OQ-139975820358-Transcript.txt,10107,MSFT,MICROSOFT CORP,2023-01-24,2023-04-25,0.008239,5,2023-01-25,2023-04-24,0.141857,62,9604,5633,3819,49,27,22,13,8,5,105,87,94,0.093750,0.010434,72,56,53,0.125000,0.010130,33,30,41,0.047619,0.011242,2,2023,Q2 2023,5.102041,4.793183,5.760670,1.353603,1.420202,1.309243


(96, 48)


## 12. Export outputs

In [14]:
# output directory
OUTPUT_DIR = Path("../../output")

OUTPUT_EVENTS = OUTPUT_DIR / "financial_event_dataset.csv"
OUTPUT_RETURNS = OUTPUT_DIR / "daily_returns_with_abret.csv"

final_df.to_csv(OUTPUT_EVENTS, index=False)
ret_small.to_csv(OUTPUT_RETURNS, index=False)

print("Saved:", OUTPUT_EVENTS.resolve())
print("Saved:", OUTPUT_RETURNS.resolve())

Saved: /Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/output/financial_event_dataset.csv
Saved: /Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/output/daily_returns_with_abret.csv


## 13. Optional: very small diagnostic sample

Use this to inspect one firm manually and verify the event windows.

In [15]:
sample_firm = final_df["company"].dropna().iloc[0]
sample_permno = final_df.loc[final_df["company"] == sample_firm, "PERMNO"].iloc[0]

print("Sample firm :", sample_firm)
print("Sample PERMNO:", sample_permno)

display(final_df[final_df["company"] == sample_firm][[
    "company", "date", "event_trading_day", "car_m1_p3",
    "long_run_start", "long_run_end", "long_run_abret"
]].sort_values("date"))

Sample firm : Microsoft Corp
Sample PERMNO: 10107


,company,date,event_trading_day,car_m1_p3,long_run_start,long_run_end,long_run_abret
0,Microsoft Corp,2022-01-25,2022-01-25,0.039141,2022-01-26,2022-04-25,-0.002991
1,Microsoft Corp,2022-04-26,2022-04-26,0.048218,2022-04-27,2022-07-25,0.017476
2,Microsoft Corp,2022-07-26,2022-07-26,0.037095,2022-07-27,2022-10-24,0.022081
3,Microsoft Corp,2022-10-25,2022-10-25,-0.061268,2022-10-26,2023-01-23,-0.069766
4,Microsoft Corp,2023-01-24,2023-01-24,0.008239,2023-01-25,2023-04-24,0.141857
5,Microsoft Corp,2023-04-25,2023-04-25,0.070937,2023-04-26,2023-07-24,0.123525
6,Microsoft Corp,2023-07-25,2023-07-25,-0.023971,2023-07-26,2023-10-23,0.028720
7,Microsoft Corp,2023-10-24,2023-10-24,0.035571,2023-10-25,2024-01-29,0.067357
8,Microsoft Corp,2024-01-30,2024-01-30,0.007434,2024-01-31,2024-04-24,-0.022996
9,Microsoft Corp,2024-04-25,2024-04-25,-0.037207,2024-04-26,2024-07-29,-0.004226
